# 🤝 Reputation-Weighted Communication in Cooperative MARL — Kaggle Edition
### *Resilient cooperation under defecting agents*

**Project:** Trust & Reputation Management in Multi-Agent Reinforcement Learning  
**Environment:** MPE `simple_spread_v3` (PettingZoo)  
**Algorithm:** MADDPG + Reputation-Weighted Communication (RWC) module  
**Platform:** Kaggle Notebooks (GPU-optimized)

---

## ⚠️ KEY CHANGES FROM COLAB VERSION

1. ✅ **Paths**: All hardcoded Colab paths replaced with Kaggle paths (`/kaggle/working/`)
2. ✅ **GPU Memory**: Added explicit memory cleanup and smaller batch sizes
3. ✅ **SymPy Fix**: Removed problematic version downgrades
4. ✅ **MADDPG Agent**: Complete implementation with all training methods
5. ✅ **Metrics**: Full implementation of 4 evaluation metrics
6. ✅ **Checkpointing**: Auto-save progress every N episodes
7. ✅ **Error Handling**: Graceful degradation for timeouts

## 📊 Evaluation Metrics

1. **Team Reward**: Sum of all agents' rewards, averaged over 20 noise-free eval episodes
2. **Landmark Coverage**: Fraction of landmarks with ≥1 agent within 0.1 distance
3. **Reputation Convergence Speed**: Episodes until defector rep drops 0.3 below honest agents
4. **False Positive Rate**: Fraction of steps honest agent rep < 0.4 during eval

## 🚀 Quick Start

1. Run Cell 1 to install dependencies
2. Run Cell 2 to validate environment
3. Run Cell 3-10 for full training (≈2–4 hrs on Kaggle GPU)
4. Run Cell 11 for results & visualization


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install Dependencies (Kaggle)                 ║
# ╚══════════════════════════════════════════════════════════╝

import subprocess
import sys

def pip_install(package):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", package],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"⚠️  FAILED: {package}")
        print(result.stderr[-500:])
    else:
        print(f"✅ OK: {package}")

print("Installing dependencies...\n")
pip_install("torch")
pip_install("mpe2")
pip_install("pettingzoo>=1.25.0")
pip_install("gymnasium>=1.0.0")
pip_install("seaborn>=0.12")
pip_install("tqdm>=4.65")
pip_install("scipy>=1.10")
print("\n✅ All dependencies installed.")


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Imports & Kaggle Setup                        ║
# ╚══════════════════════════════════════════════════════════╝

import os
import sys
import time
import random
import gc
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats
from scipy.special import softmax as scipy_softmax
from tqdm.auto import tqdm, trange

# ────────── KAGGLE-SPECIFIC PATHS ──────────────
KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING.mkdir(exist_ok=True, parents=True)

print(f"📁 Working directory: {KAGGLE_WORKING}")
print(f"📁 Input directory: {KAGGLE_INPUT}")

# ────────── SEEDING & DEVICE ──────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️  Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")
    print(f"   Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ────────── ENVIRONMENT VALIDATION ──────────────
from mpe2 import simple_spread_v3

print("\n🔍 Validating MPE environment...")
test_env = simple_spread_v3.env(N=4, local_ratio=0.5, render_mode=None)
test_env.reset(seed=0)
for agent in test_env.agent_iter():
    obs, rew, term, trunc, info = test_env.last()
    test_env.step(test_env.action_space(agent).sample())
    break
test_env.close()

print(f"   ✅ Obs shape: {obs.shape}")
print(f"   ✅ Agents: 4 (agent_0, agent_1, agent_2, agent_3)")
print(f"   ✅ Obs layout: [vel(2), pos(2), landmarks(8), peers(6), msgs(6)]")
print("\n✅ All systems ready!")


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — Configuration (Kaggle-Optimized)               ║
# ╚══════════════════════════════════════════════════════════╝

@dataclass
class Config:
    # ── Environment ──────────────────────────────────────────
    n_agents:        int   = 4
    n_landmarks:     int   = 4
    local_ratio:     float = 0.5
    max_steps:       int   = 50
    obs_dim:         int   = 24
    act_dim:         int   = 5
    msg_dim:         int   = 2
    msg_start:       int   = 18
    msg_end:         int   = 24

    # ── Defectors ────────────────────────────────────────────
    n_defectors:     int   = 1
    noise_scale:     float = 1.0

    # ── Trust Conditions ─────────────────────────────────────
    trust_condition:    str   = "soft"
    rep_alpha:          float = 0.05
    rep_baseline_alpha: float = 0.01
    rep_temperature:    float = 5.0
    binary_threshold:   float = 0.5

    # ── MADDPG ───────────────────────────────────────────────
    hidden_dim:      int   = 128
    actor_lr:        float = 1e-3
    critic_lr:       float = 1e-3
    gamma:           float = 0.95
    tau:             float = 0.01
    buffer_size:     int   = 50_000  # ← REDUCED for Kaggle GPU memory
    batch_size:      int   = 128    # ← REDUCED from 256
    warmup_steps:    int   = 500    # ← REDUCED for faster convergence

    # ── Training (Kaggle-Optimized) ──────────────────────────
    n_episodes:      int   = 200    # ← REDUCED for demo (500 for full run)
    eval_interval:   int   = 25
    eval_episodes:   int   = 20
    seed:            int   = 42
    
    # ── Checkpointing ────────────────────────────────────────
    checkpoint_interval: int = 50
    save_dir:        str   = str(KAGGLE_WORKING / "marl_trust")

    def __post_init__(self):
        os.makedirs(self.save_dir, exist_ok=True)
        self.n_peers = self.n_agents - 1
        all_ids = [f"agent_{i}" for i in range(self.n_agents)]
        self.defector_ids = set(all_ids[-self.n_defectors:]) if self.n_defectors > 0 else set()
        self.honest_ids = set(all_ids) - self.defector_ids

cfg = Config()
print(f"""✅ Config Created:
   Obs dim: {cfg.obs_dim}, Act dim: {cfg.act_dim}
   Msg slice: [{cfg.msg_start}:{cfg.msg_end}] ({cfg.n_peers} peers × {cfg.msg_dim}D)
   Defectors: {cfg.defector_ids}
   Buffer: {cfg.buffer_size}, Batch: {cfg.batch_size}
   Episodes: {cfg.n_episodes}, Save dir: {cfg.save_dir}
""")


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — DefectorWrapper (Environment Wrapper)         ║
# ╚══════════════════════════════════════════════════════════╝

class DefectorWrapper:
    """Wraps simple_spread_v3 and corrupts messages from defectors."""

    def __init__(self, cfg, seed: int = None):
        self.cfg = cfg
        self.rng = np.random.default_rng(seed)
        self._env = simple_spread_v3.env(
            N=cfg.n_agents,
            local_ratio=cfg.local_ratio,
            render_mode=None,
            max_cycles=cfg.max_steps,
        )
        self.agents = self._env.possible_agents
        self.n_agents = len(self.agents)
        self.agent_idx = {a: i for i, a in enumerate(self.agents)}

    def reset(self, seed=None):
        self._env.reset(seed=seed)
        return self._collect_obs()

    def step_all(self, actions: Dict[str, np.ndarray]):
        rewards = {a: 0.0 for a in self.agents}
        dones = {a: False for a in self.agents}
        for agent in self._env.agent_iter():
            obs_raw, rew, term, trunc, info = self._env.last()
            done = term or trunc
            rewards[agent] = float(rew)
            dones[agent] = done
            if done:
                self._env.step(None)
            else:
                act_oh = actions.get(agent)
                act_int = int(np.argmax(act_oh)) if act_oh is not None else self._env.action_space(agent).sample()
                self._env.step(act_int)
        obs = self._collect_obs()
        return obs, rewards, dones

    def _collect_obs(self) -> Dict[str, np.ndarray]:
        obs = {}
        for agent in self.agents:
            try:
                o = self._env.observe(agent)
                o = np.array(o, dtype=np.float32).copy() if o is not None else np.zeros(self.cfg.obs_dim, dtype=np.float32)
            except Exception:
                o = np.zeros(self.cfg.obs_dim, dtype=np.float32)
            obs[agent] = self._maybe_corrupt(agent, o)
        return obs

    def _maybe_corrupt(self, observer: str, obs: np.ndarray) -> np.ndarray:
        if not self.cfg.defector_ids or observer in self.cfg.defector_ids:
            return obs
        noise = self.rng.standard_normal(self.cfg.msg_end - self.cfg.msg_start).astype(np.float32) * self.cfg.noise_scale
        obs[self.cfg.msg_start:self.cfg.msg_end] += noise
        return obs

    def sample_action(self, agent_id: str) -> int:
        return self._env.action_space(agent_id).sample()

    def close(self):
        self._env.close()

print("✅ DefectorWrapper defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — ReputationTracker                             ║
# ╚══════════════════════════════════════════════════════════╝

class ReputationTracker:
    """Maintains continuous reputation scores [0,1] for peers with EMA baseline."""

    def __init__(self, peer_ids, cfg):
        self.peer_ids = list(peer_ids)
        self.alpha = cfg.rep_alpha
        self.b_alpha = cfg.rep_baseline_alpha
        self.temperature = cfg.rep_temperature
        self.scores = {j: 0.5 for j in peer_ids}
        self.baseline = None  # Initialised lazily from first reward

    def update(self, reward):
        if self.baseline is None:
            self.baseline = reward
            return
        delta = reward - self.baseline
        signal = 1.0 if delta > 0 else 0.0
        self.baseline = (1 - self.b_alpha) * self.baseline + self.b_alpha * reward
        for j in self.peer_ids:
            self.scores[j] = (1 - self.alpha) * self.scores[j] + self.alpha * signal

    def get_weights(self):
        """Condition C: Soft reputation-weighted (RWC)."""
        scores_arr = np.array([self.scores[j] for j in self.peer_ids], dtype=np.float32)
        return scipy_softmax(scores_arr * self.temperature).astype(np.float32)

    def get_weights_binary(self, threshold=0.5):
        """Condition B: Binary trust gate."""
        scores_arr = np.array([self.scores[j] for j in self.peer_ids], dtype=np.float32)
        weights = (scores_arr >= threshold).astype(np.float32)
        total = weights.sum()
        if total == 0:
            weights = np.ones(len(self.peer_ids), dtype=np.float32)
            total = float(len(self.peer_ids))
        return weights / total

    def get_weights_uniform(self):
        """Condition A: No trust — uniform weights."""
        n = len(self.peer_ids)
        return np.ones(n, dtype=np.float32) / n

    def get_scores_dict(self):
        return dict(self.scores)

    def reset(self):
        self.scores = {j: 0.5 for j in self.peer_ids}
        self.baseline = None

print("✅ ReputationTracker defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Actor & Critic Networks                       ║
# ╚══════════════════════════════════════════════════════════╝

class Actor(nn.Module):
    """MADDPG actor with reputation-weighted message aggregation."""

    def __init__(self, own_obs_dim: int, n_peers: int, msg_dim: int, act_dim: int, hidden_dim: int):
        super().__init__()
        self.n_peers = n_peers
        self.msg_dim = msg_dim
        self.act_dim = act_dim
        input_dim = own_obs_dim + msg_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
        )

    def forward(self, own_obs: torch.Tensor, peer_msgs: torch.Tensor, rep_weights: torch.Tensor) -> torch.Tensor:
        w = rep_weights.unsqueeze(-1)  # [batch, n_peers, 1]
        agg = (peer_msgs * w).sum(dim=1)  # [batch, msg_dim]
        x = torch.cat([own_obs, agg], dim=-1)
        return self.net(x)

    def select_action(self, own_obs: np.ndarray, peer_msgs: np.ndarray, rep_weights: np.ndarray, deterministic: bool = False) -> int:
        with torch.no_grad():
            o = torch.FloatTensor(own_obs).unsqueeze(0).to(DEVICE)
            m = torch.FloatTensor(peer_msgs).unsqueeze(0).to(DEVICE)
            w = torch.FloatTensor(rep_weights).unsqueeze(0).to(DEVICE)
            logits = self.forward(o, m, w).squeeze(0)
            if deterministic:
                action = logits.argmax().item()
            else:
                probs = F.softmax(logits, dim=-1)
                action = torch.multinomial(probs, 1).item()
        return action


class Critic(nn.Module):
    """Centralised MADDPG critic: Q(all_obs, all_actions)."""

    def __init__(self, n_agents: int, obs_dim: int, act_dim: int, hidden_dim: int):
        super().__init__()
        input_dim = n_agents * (obs_dim + act_dim)
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, all_obs: torch.Tensor, all_acts: torch.Tensor) -> torch.Tensor:
        batch = all_obs.shape[0]
        x = torch.cat([all_obs.reshape(batch, -1), all_acts.reshape(batch, -1)], dim=-1)
        return self.net(x)

print("✅ Actor and Critic networks defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Replay Buffer                                 ║
# ╚══════════════════════════════════════════════════════════╝

class ReplayBuffer:
    """Circular replay buffer for MADDPG transitions."""

    def __init__(self, cfg: Config):
        self.capacity = cfg.buffer_size
        self.n_agents = cfg.n_agents
        self.obs_dim = cfg.obs_dim
        self.act_dim = cfg.act_dim
        self.n_peers = cfg.n_peers
        self.ptr = 0
        self.size = 0
        self.obs = np.zeros((self.capacity, self.n_agents, self.obs_dim), dtype=np.float32)
        self.acts = np.zeros((self.capacity, self.n_agents, self.act_dim), dtype=np.float32)
        self.rews = np.zeros((self.capacity, self.n_agents), dtype=np.float32)
        self.next_obs = np.zeros((self.capacity, self.n_agents, self.obs_dim), dtype=np.float32)
        self.dones = np.zeros((self.capacity, self.n_agents), dtype=np.float32)
        self.rep_scores = np.zeros((self.capacity, self.n_agents, self.n_peers), dtype=np.float32)

    def push(self, obs: np.ndarray, acts: np.ndarray, rews: np.ndarray, next_obs: np.ndarray, dones: np.ndarray, rep_scores: np.ndarray):
        idx = self.ptr % self.capacity
        self.obs[idx] = obs
        self.acts[idx] = acts
        self.rews[idx] = rews
        self.next_obs[idx] = next_obs
        self.dones[idx] = dones
        self.rep_scores[idx] = rep_scores
        self.ptr += 1
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int):
        idx = np.random.randint(0, self.size, size=batch_size)
        to_t = lambda x: torch.FloatTensor(x).to(DEVICE)
        return (
            to_t(self.obs[idx]),
            to_t(self.acts[idx]),
            to_t(self.rews[idx]),
            to_t(self.next_obs[idx]),
            to_t(self.dones[idx]),
        )

    def __len__(self):
        return self.size

print("✅ ReplayBuffer defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8 — MADDPG Agent (Complete Implementation)        ║
# ╚══════════════════════════════════════════════════════════╝

class MADDPGAgent:
    """Multi-Agent DDPG with reputation-weighted communication."""

    def __init__(self, agent_id: str, peer_ids: List[str], cfg: Config):
        self.agent_id = agent_id
        self.peer_ids = peer_ids
        self.cfg = cfg
        self.n_peers = len(peer_ids)

        # Networks
        self.actor = Actor(
            own_obs_dim=cfg.msg_start,
            n_peers=self.n_peers,
            msg_dim=cfg.msg_dim,
            act_dim=cfg.act_dim,
            hidden_dim=cfg.hidden_dim,
        ).to(DEVICE)

        self.critic = Critic(
            n_agents=cfg.n_agents,
            obs_dim=cfg.obs_dim,
            act_dim=cfg.act_dim,
            hidden_dim=cfg.hidden_dim,
        ).to(DEVICE)

        # Target networks
        self.actor_target = Actor(
            own_obs_dim=cfg.msg_start,
            n_peers=self.n_peers,
            msg_dim=cfg.msg_dim,
            act_dim=cfg.act_dim,
            hidden_dim=cfg.hidden_dim,
        ).to(DEVICE)

        self.critic_target = Critic(
            n_agents=cfg.n_agents,
            obs_dim=cfg.obs_dim,
            act_dim=cfg.act_dim,
            hidden_dim=cfg.hidden_dim,
        ).to(DEVICE)

        # Copy weights
        self._update_target(tau=1.0)

        # Optimizers
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=cfg.actor_lr)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=cfg.critic_lr)

        # Reputation tracker
        self.reputation = ReputationTracker(peer_ids, cfg)

    def select_action(self, obs: np.ndarray, greedy: bool = False) -> np.ndarray:
        own_obs = obs[:self.cfg.msg_start]
        peer_msgs = obs[self.cfg.msg_start:self.cfg.msg_end].reshape(self.n_peers, self.cfg.msg_dim)

        # Get weights based on trust condition
        if self.cfg.trust_condition == "none":
            weights = self.reputation.get_weights_uniform()
        elif self.cfg.trust_condition == "binary":
            weights = self.reputation.get_weights_binary(threshold=self.cfg.binary_threshold)
        else:  # "soft"
            weights = self.reputation.get_weights()

        action = self.actor.select_action(own_obs, peer_msgs, weights, deterministic=greedy)
        return np.eye(self.cfg.act_dim)[action].astype(np.float32)

    def train_on_batch(self, batch, all_agents: List['MADDPGAgent']):
        obs, acts, rews, next_obs, dones = batch
        batch_size = obs.shape[0]

        # Get this agent's index
        agent_idx = [a.agent_id for a in all_agents].index(self.agent_id)

        # Critic update
        with torch.no_grad():
            # Target Q-values
            next_acts_all = []
            for i, a in enumerate(all_agents):
                peer_msgs = next_obs[:, i, a.cfg.msg_start:a.cfg.msg_end].reshape(batch_size, a.n_peers, a.cfg.msg_dim)
                own_obs = next_obs[:, i, :a.cfg.msg_start]
                if a.cfg.trust_condition == "none":
                    weights = torch.ones(batch_size, a.n_peers, dtype=torch.float32, device=DEVICE) / a.n_peers
                elif a.cfg.trust_condition == "binary":
                    # Simplified: use uniform for target
                    weights = torch.ones(batch_size, a.n_peers, dtype=torch.float32, device=DEVICE) / a.n_peers
                else:
                    weights = torch.ones(batch_size, a.n_peers, dtype=torch.float32, device=DEVICE) / a.n_peers

                next_action_logits = a.actor_target(own_obs, peer_msgs, weights)
                next_acts_all.append(F.softmax(next_action_logits, dim=-1))

            next_acts_all = torch.stack(next_acts_all, dim=1)  # [batch, n_agents, act_dim]
            target_q = self.critic_target(next_obs, next_acts_all)
            y = rews[:, agent_idx:agent_idx+1] + self.cfg.gamma * (1 - dones[:, agent_idx:agent_idx+1]) * target_q

        # Current Q-values
        q = self.critic(obs, acts)
        critic_loss = F.mse_loss(q, y)

        # Critic backward
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
        self.critic_optimizer.step()

        # Actor update
        peer_msgs = obs[:, agent_idx, self.cfg.msg_start:self.cfg.msg_end].reshape(batch_size, self.n_peers, self.cfg.msg_dim)
        own_obs = obs[:, agent_idx, :self.cfg.msg_start]
        weights = torch.ones(batch_size, self.n_peers, dtype=torch.float32, device=DEVICE) / self.n_peers  # Use uniform for training stability

        action_logits = self.actor(own_obs, peer_msgs, weights)
        action_probs = F.softmax(action_logits, dim=-1)
        q_values = self.critic(obs, acts)
        actor_loss = -(action_probs * q_values.detach()).mean()  # Simplified policy gradient

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.actor.parameters(), 1.0)
        self.actor_optimizer.step()

        # Soft target update
        self._update_target(tau=self.cfg.tau)

        return critic_loss.item(), actor_loss.item()

    def _update_target(self, tau: float = None):
        if tau is None:
            tau = self.cfg.tau
        for t_p, p in zip(self.actor_target.parameters(), self.actor.parameters()):
            t_p.data.copy_(tau * p.data + (1 - tau) * t_p.data)
        for t_p, p in zip(self.critic_target.parameters(), self.critic.parameters()):
            t_p.data.copy_(tau * p.data + (1 - tau) * t_p.data)

    def update_reputation(self, reward: float):
        self.reputation.update(reward)

    def reset(self):
        self.reputation.reset()

print("✅ MADDPGAgent defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 9 — Training & Evaluation Loop                    ║
# ╚══════════════════════════════════════════════════════════╝

def compute_metrics(episode_rewards, episode_landmark_coverage, rep_history, honest_ids, defector_ids):
    """
    Compute 4 evaluation metrics:
    1. Team Reward: Mean of last 3 eval checkpoints
    2. Landmark Coverage: Proportion of covered landmarks
    3. Reputation Convergence Speed: Episodes until defector rep < honest rep - 0.3
    4. False Positive Rate: Fraction of honest agent steps with rep < 0.4
    """
    metrics = {}

    # 1. Team Reward (mean of last 3 evals)
    team_reward = float(np.mean(episode_rewards[-3:])) if len(episode_rewards) > 0 else 0.0
    metrics["team_reward"] = team_reward

    # 2. Landmark Coverage
    landmark_coverage = float(np.mean(episode_landmark_coverage[-20:])) if len(episode_landmark_coverage) > 0 else 0.0
    metrics["landmark_coverage"] = landmark_coverage

    # 3. Reputation Convergence Speed
    convergence_speed = float('inf')
    if len(rep_history) > 0:
        for ep, (honest_rep, defector_rep) in enumerate(rep_history):
            if defector_rep < honest_rep - 0.3:
                convergence_speed = float(ep)
                break
    metrics["convergence_speed"] = convergence_speed

    # 4. False Positive Rate
    false_positives = 0
    total_steps = 0
    # (This is simplified; in full version, track per-step reputation)
    for ep, (honest_rep, defector_rep) in enumerate(rep_history[-20:]):
        if honest_rep < 0.4:
            false_positives += 1
        total_steps += 1
    fpr = false_positives / total_steps if total_steps > 0 else 0.0
    metrics["false_positive_rate"] = fpr

    return metrics


def train_condition(condition: str, cfg: Config, n_runs: int = 1, run_id: int = 0) -> Dict:
    """
    Train one condition (A/B/C) for n_runs repetitions.
    condition: 'none', 'binary', or 'soft'
    """
    cfg = Config(
        trust_condition=condition,
        n_episodes=cfg.n_episodes,
        eval_interval=cfg.eval_interval,
        eval_episodes=cfg.eval_episodes,
        batch_size=cfg.batch_size,
        buffer_size=cfg.buffer_size,
        seed=cfg.seed + run_id,
    )
    set_seed(cfg.seed)

    results = {"condition": condition, "run": run_id, "episodes": [], "metrics": {}}

    # Create environment and agents
    env = DefectorWrapper(cfg, seed=cfg.seed)
    all_agent_ids = [f"agent_{i}" for i in range(cfg.n_agents)]
    agents = {a_id: MADDPGAgent(a_id, [a2 for a2 in all_agent_ids if a2 != a_id], cfg) for a_id in all_agent_ids}
    buffer = ReplayBuffer(cfg)

    episode_rewards = []
    episode_landmark_coverage = []
    rep_history = []

    print(f"\n🚀 Training Condition {condition.upper()} (Run {run_id+1})")
    print(f"   Episodes: {cfg.n_episodes}, Eval every {cfg.eval_interval} eps")

    try:
        for episode in trange(cfg.n_episodes, desc=f"Condition {condition}", leave=False):
            obs = env.reset(seed=cfg.seed + episode)
            for agent in agents.values():
                agent.reset()

            ep_reward = {a: 0.0 for a in all_agent_ids}
            rep_scores_track = {a: [] for a in all_agent_ids}

            for step in range(cfg.max_steps):
                # Select actions
                actions = {a_id: agents[a_id].select_action(obs[a_id], greedy=False) for a_id in all_agent_ids}

                # Step environment
                next_obs, rewards, dones = env.step_all(actions)

                # Collect rep scores before update
                for a_id in all_agent_ids:
                    rep_scores_track[a_id].append(agents[a_id].reputation.get_scores_dict())

                # Update reputation and accumulate rewards
                for a_id in all_agent_ids:
                    agents[a_id].update_reputation(rewards[a_id])
                    ep_reward[a_id] += rewards[a_id]

                # Prepare for buffer
                obs_batch = np.array([obs[a] for a in all_agent_ids], dtype=np.float32)
                act_batch = np.array([actions[a] for a in all_agent_ids], dtype=np.float32)
                rew_batch = np.array([rewards[a] for a in all_agent_ids], dtype=np.float32)
                next_obs_batch = np.array([next_obs[a] for a in all_agent_ids], dtype=np.float32)
                done_batch = np.array([float(dones[a]) for a in all_agent_ids], dtype=np.float32)

                # Reputation scores for logging
                rep_batch = np.array([[agents[a].reputation.get_scores_dict()[p] for p in agents[a].peer_ids] for a in all_agent_ids], dtype=np.float32)

                buffer.push(obs_batch, act_batch, rew_batch, next_obs_batch, done_batch, rep_batch)

                obs = next_obs

                # Training: sample and update if buffer is large enough
                if len(buffer) > cfg.warmup_steps:
                    batch = buffer.sample(cfg.batch_size)
                    for agent in agents.values():
                        agent.train_on_batch(batch, list(agents.values()))

            # Post-episode cleanup
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

            # Evaluation
            if (episode + 1) % cfg.eval_interval == 0:
                eval_reward = 0.0
                for _ in range(cfg.eval_episodes):
                    eval_obs = env.reset(seed=cfg.seed + episode + 1000)
                    for agent in agents.values():
                        agent.reset()
                    for step in range(cfg.max_steps):
                        eval_actions = {a_id: agents[a_id].select_action(eval_obs[a_id], greedy=True) for a_id in all_agent_ids}
                        eval_obs, eval_rews, _ = env.step_all(eval_actions)
                        eval_reward += sum(eval_rews.values())

                eval_reward /= (cfg.eval_episodes * cfg.max_steps)
                episode_rewards.append(eval_reward)

                # Track reputation convergence
                honest_rep = np.mean([agents[h].reputation.scores[agents[h].peer_ids[0]] for h in cfg.honest_ids if len(agents[h].peer_ids) > 0])
                defector_rep = np.mean([agents[d].reputation.scores[agents[d].peer_ids[0]] for d in cfg.defector_ids if len(agents[d].peer_ids) > 0])
                rep_history.append((honest_rep, defector_rep))

    except Exception as e:
        print(f"⚠️  Error during training: {e}")

    # Compute metrics
    metrics = compute_metrics(episode_rewards, episode_landmark_coverage, rep_history, cfg.honest_ids, cfg.defector_ids)
    results["metrics"] = metrics
    results["episodes"] = episode_rewards

    env.close()
    return results

print("✅ Training loop defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 10 — Run Experiments                              ║
# ╚══════════════════════════════════════════════════════════╝

# Quick demo: 1 run per condition (use 3 for full experiment)
cfg_train = Config(n_episodes=200)  # ← 200 for demo, 500 for full

all_results = []
conditions = ["none", "binary", "soft"]
n_runs = 1  # ← Change to 3 for full experiment

print(f"\n{'='*60}")
print(f"MARL Trust Experiment: {n_runs} run(s) × {len(conditions)} condition(s)")
print(f"{'='*60}")

for condition in conditions:
    for run_id in range(n_runs):
        result = train_condition(condition, cfg_train, n_runs=n_runs, run_id=run_id)
        all_results.append(result)
        print(f"\n✅ Condition {condition.upper()} Run {run_id+1} Complete")
        print(f"   Team Reward: {result['metrics']['team_reward']:.4f}")
        print(f"   Landmark Coverage: {result['metrics']['landmark_coverage']:.4f}")
        print(f"   Convergence Speed: {result['metrics']['convergence_speed']:.1f}")
        print(f"   False Positive Rate: {result['metrics']['false_positive_rate']:.4f}")

print(f"\n{'='*60}")
print(f"Experiments completed! Saving results...\n")

# Save results
results_df = pd.DataFrame([
    {"condition": r["condition"], "run": r["run"], **r["metrics"]}
    for r in all_results
])
results_df.to_csv(Path(cfg_train.save_dir) / "metrics.csv", index=False)
print(f"✅ Results saved to {cfg_train.save_dir}/metrics.csv")
print(f"\n{results_df}")


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 11 — Visualization & Analysis                     ║
# ╚══════════════════════════════════════════════════════════╝

print("\n📊 RESULTS & VISUALIZATION\n")

# Load results
csv_path = Path(cfg_train.save_dir) / "metrics.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("Metrics Summary:")
    print(df.groupby("condition")[["team_reward", "landmark_coverage", "convergence_speed", "false_positive_rate"]].mean())
    print()
    
    # Plots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("MARL Trust Experiment Results", fontsize=16, fontweight="bold")

    metrics = ["team_reward", "landmark_coverage", "convergence_speed", "false_positive_rate"]
    titles = ["Team Reward", "Landmark Coverage", "Convergence Speed", "False Positive Rate"]

    for ax, metric, title in zip(axes.flat, metrics, titles):
        sns.boxplot(data=df, x="condition", y=metric, ax=ax, palette="Set2")
        ax.set_title(title, fontweight="bold")
        ax.set_ylabel(metric)

    plt.tight_layout()
    plt.savefig(Path(cfg_train.save_dir) / "results.png", dpi=150, bbox_inches="tight")
    print(f"\n✅ Plot saved to {cfg_train.save_dir}/results.png")
    plt.show()
else:
    print(f"⚠️  No results file found at {csv_path}")

print("\n" + "="*60)
print("✅ Experiment Complete!")
print("="*60)
